# Feature Samples Staging Load

In [ ]:
%run "./initialize"

## Base Source Data

In [ ]:
spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_customer (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, DELETE_FLAG, LOAD_TIMESTAMP)
VALUES
  (1, 'John', 'Doe', 'john.doe@example.com', NULL, '2023-01-01 10:00:00'),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', NULL, '2023-01-01 10:00:00'),
  (10, 'Richard', 'Johnson', 'richard.johnson@example.com', NULL, '2023-01-01 10:00:00')""")

spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_customer_address (
  CUSTOMER_ID, CITY, STATE, LOAD_TIMESTAMP)
VALUES
  (1, 'Melbourne', 'VIC', '2023-01-01 10:00:00'),
  (2, 'Melbourne', 'VIC', '2023-01-01 10:00:00'),
  (NULL, 'Melbourne', 'VIC', '2023-01-01 10:00:00'),
  (4, 'Hobart', 'TAS', '2023-01-01 10:00:00'),
  (10, 'Sydney', 'NSW', '2023-01-01 10:00:00')""")

spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_customer_purchase (
  CUSTOMER_ID, PRODUCT, QUANTITY, PRICE, PURCHASE_TIMESTAMP)
VALUES
  (1, 'Apples', 1, 10.00, '2023-01-01 10:00:00'),
  (1, 'Bananas', 2, 20.00, '2023-01-01 10:00:00'),
  (2, 'Oranges', 3, 30.00, '2023-01-01 10:00:00'),
  (2, 'Pears', 4, 40.00, '2023-01-01 10:00:00'),
  (10, 'Apples', 5, 50.00, '2023-01-01 10:00:00'),
  (10, 'Bananas', 6, 60.00, '2023-01-01 10:00:00')""")

## File Sources

In [ ]:
file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
1,John,Doe,john.doe@example.com,,2023-01-01 10:00:00\n
2,Jane,Smith,jane.smith@example.com,,2023-01-01 10:00:00\n
"""
dbutils.fs.put(f"{customer_file_path}/customer_1.csv", file_content, True)

## Snapshot Sources

In [ ]:
spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_customer_snapshot_source (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, DELETE_FLAG, LOAD_TIMESTAMP)
VALUES
  (1, 'John', 'Doe', 'john.doe@example.com', NULL, '2023-01-01 10:00:00'),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', NULL, '2023-01-01 10:00:00'),
  (10, 'Richard', 'Johnson', 'richard.johnson@example.com', NULL, '2023-01-01 10:00:00')""")

spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_customer_historical_snapshot_source (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, LOAD_TIMESTAMP)
VALUES
  (1, 'John', 'Doe', 'john.doe@example.com', '2024-01-01 10:00:00'),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', '2024-01-01 10:00:00'),
  (1, 'John', 'Doe', 'jdoe@example.com', '2024-01-04 10:00:00'),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', '2024-01-04 10:00:00'),
  (3, 'Alice', 'Green', 'alice.green@example.com', '2024-01-04 10:00:00'),
  (4, 'Joe', 'Bloggs', 'joe.bloggs@example.com', '2023-01-04 10:00:00'),
  (1, 'John', 'Doe', 'jdoe@example.com', '2024-02-10 10:00:00'),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', '2024-02-10 10:00:00'),
  (3, 'Alice', 'Green', 'alice.green@example.com', '2024-02-10 10:00:00')""")

spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_customer_snapshots (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, UPDATE_TIMESTAMP, SNAPSHOT_TIMESTAMP, SNAPSHOT_VERSION)
VALUES
  (1, 'John', 'Doe', 'john.doe@example.com', '2023-01-01 00:00:00', '2023-01-01 00:00:00', 0),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', '2023-01-01 00:00:00', '2023-01-01 00:00:00', 0),
  (10, 'Richard', 'Johnson', 'richard.johnson@example.com', '2023-01-01 00:00:00', '2023-01-01 00:00:00', 0)""")

## Snapshot File Sources

In [ ]:
dbutils.fs.rm(customer_snapshot_file_path, True)
dbutils.fs.rm(customer_snapshot_regex_file_path, True)
dbutils.fs.rm(customer_snapshot_partitioned_file_path, True)
dbutils.fs.rm(customer_snapshot_partitioned_parquet_file_path, True)
dbutils.fs.rm(template_samples_customer_file_path, True)
dbutils.fs.rm(template_samples_customer_address_file_path, True)

file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
1,John,Doe,john.doe@example.com,,2024-01-01 10:00:00\n
2,Jane,Smith,jane.smith@example.com,,2024-01-01 10:00:00\n
"""
dbutils.fs.put(f"{customer_snapshot_file_path}/customer_2024_01_01.csv", file_content, True)
dbutils.fs.put(f"{customer_snapshot_regex_file_path}/2024/01/data/customer_01.csv", file_content, True)
dbutils.fs.put(f"{customer_snapshot_partitioned_file_path}/YEAR=2024/MONTH=01/DAY=01/customer.csv", file_content, True)
dbutils.fs.put(f"{template_samples_customer_file_path}/customer_2024_01_01.csv", file_content, True)

file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
1,John,Doe,jdoe@example.com,,2024-01-04 10:00:00\n
3,Alice,Green,alice.green@example.com,,2024-01-04 10:00:00\n
4,Joe,Bloggs,joe.bloggs@example.com,,2024-01-04 10:00:00\n
"""
dbutils.fs.put(f"{customer_snapshot_file_path}/customer_2024_01_04.csv", file_content, True)
dbutils.fs.put(f"{customer_snapshot_file_path}/sub_dir_test/customer_2024_01_04.csv", file_content, True)
dbutils.fs.put(f"{customer_snapshot_partitioned_file_path}/YEAR=2024/MONTH=01/DAY=04/customer.csv", file_content, True)
dbutils.fs.put(f"{template_samples_customer_file_path}/customer_2024_01_04.csv", file_content, True)

file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
1,John,Doe,jdoe@example.com,,2024-02-10 10:00:00\n
3,Alice,Green,alice.green@example.com,,2024-02-10 10:00:00\n
4,Joe,Bloggs,joe.bloggs@example.com,,2024-02-10 10:00:00\n
5,Sarah,Jones,sarah.jones@example.com,,2024-02-10 10:00:00\n
"""
dbutils.fs.put(f"{customer_snapshot_file_path}/customer_2024_02_10.csv", file_content, True)
dbutils.fs.put(f"{customer_snapshot_partitioned_file_path}/YEAR=2024/MONTH=02/DAY=10/customer.csv", file_content, True)
dbutils.fs.put(f"{template_samples_customer_file_path}/customer_2024_02_10.csv", file_content, True)

file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
1,John,Doe,john.doe@example.com,,2024-01-01 10:00:00\n
"""
dbutils.fs.put(f"{customer_snapshot_multifile_path}/customer_2024_01_01_split_0001.csv", file_content, True)
file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
2,Jane,Smith,jane.smith@example.com,,2024-01-01 10:00:00\n
"""
dbutils.fs.put(f"{customer_snapshot_multifile_path}/customer_2024_01_01_split_0002.csv", file_content, True)
file_content = """CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,DELETE_FLAG,LOAD_TIMESTAMP\n
1,John,Doe,john@example.com,,2024-12-12 10:00:00\n
"""
dbutils.fs.put(f"{customer_snapshot_multifile_path}/customer_2024_12_12_split_0001.csv", file_content, True)

In [ ]:
# Historical Snapshot File Sources for customer_address

customer_address_file_content = """CUSTOMER_ID,CITY,STATE,LOAD_TIMESTAMP
1,Melbourne,VIC,2024-01-01 10:00:00
2,Melbourne,VIC,2024-01-01 10:00:00
"""
dbutils.fs.put(f"{template_samples_customer_address_file_path}/customer_address_2024_01_01.csv", customer_address_file_content, True)

customer_address_file_content = """CUSTOMER_ID,CITY,STATE,LOAD_TIMESTAMP
1,Melbourne,VIC,2024-01-04 10:00:00
4,Hobart,TAS,2024-01-04 10:00:00
10,Sydney,NSW,2024-01-04 10:00:00
"""
dbutils.fs.put(f"{template_samples_customer_address_file_path}/customer_address_2024_01_04.csv", customer_address_file_content, True)

customer_address_file_content = """CUSTOMER_ID,CITY,STATE,LOAD_TIMESTAMP
1,Sydney,NSW,2024-02-10 10:00:00
4,Hobart,TAS,2024-02-10 10:00:00
10,Brisbane,QLD,2024-02-10 10:00:00
12,Perth,WA,2024-02-10 10:00:00
"""
dbutils.fs.put(f"{template_samples_customer_address_file_path}/customer_address_2024_02_10.csv", customer_address_file_content, True)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

# Historical Snapshot Parquet Sources
schema = StructType([
    StructField("CUSTOMER_ID", StringType(), True),
    StructField("FIRST_NAME", StringType(), True),
    StructField("LAST_NAME", StringType(), True),
    StructField("EMAIL", StringType(), True),
    StructField("DELETE_FLAG", StringType(), True),
    StructField("LOAD_TIMESTAMP", StringType(), True)
])

data = [
    ["1", "John", "Doe", "john.doe@example.com", "", "2024-01-01 10:00:00"],
    ["2", "Jane", "Smith", "jane.smith@example.com", "", "2024-01-01 10:00:00"],
    ["1", "John", "Doe", "john.doe@example.com", "", "2024-01-01 10:00:00"]
]
df = spark.createDataFrame(data, schema=schema)
df.write.parquet(
    f"{customer_snapshot_partitioned_parquet_file_path}/YEAR=2024/MONTH=01/DAY=01/customer.parquet",
    mode="overwrite"
)

data = [
    ["1", "John", "Doe", "jdoe@example.com", "", "2024-01-04 10:00:00"],
    ["3", "Alice", "Green", "alice.green@example.com", "", "2024-01-04 10:00:00"],
    ["4", "Joe", "Bloggs", "joe.bloggs@example.com", "", "2024-01-04 10:00:00"]
]
df = spark.createDataFrame(data, schema=schema)
df.write.parquet(
    f"{customer_snapshot_partitioned_parquet_file_path}/YEAR=2024/MONTH=01/DAY=04/customer.parquet",
    mode="overwrite"
)

data = [
    ["1", "John", "Doe", "jdoe@example.com", "", "2024-02-10 10:00:00"],
    ["3", "Alice", "Green", "alice.green@example.com", "", "2024-02-10 10:00:00"],
    ["4", "Joe", "Bloggs", "joe.bloggs@example.com", "", "2024-02-10 10:00:00"],
    ["5", "Sarah", "Jones", "sarah.jones@example.com", "", "2024-02-10 10:00:00"]
]
df = spark.createDataFrame(data, schema=schema)
df.write.parquet(
    f"{customer_snapshot_partitioned_parquet_file_path}/YEAR=2024/MONTH=02/DAY=10/customer.parquet",
    mode="overwrite"
)

## Migration Source Tables

In [ ]:
spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_table_to_migrate_scd2 (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, EFFECTIVE_FROM, EFFECTIVE_TO)
VALUES
  (1, 'John', 'Doe', 'john.doe@example.com', '2023-01-01 10:00:00', NULL),
  (2, 'Jane', 'Smith', 'jane.smith@example.com', '2023-01-01 10:00:00', NULL),
  (10, 'Richard', 'Johnson', 'richard.johnson@example.com', '2023-01-01 10:00:00', NULL),
  (30, 'Closed Same Day', 'Customer', 'closed_same_day.customer@example.com', '2023-01-01 10:00:00', '2023-01-01 10:00:00'),
  (40, 'Closed Normal', 'Customer', 'cnormal.customer@example.com', '2023-01-02 10:00:00', '2023-05-01 10:00:00'),
  (40, 'Closed Normal', 'Customer', 'closed_normal.customer@example.com', '2023-01-01 10:00:00', '2023-01-02 10:00:00')""")

spark.sql(f"""INSERT INTO TABLE {feature_schema}.src_table_to_migrate_scd0 (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL)
VALUES
  (1, 'John', 'Doe', 'john.doe@example.com'),
  (2, 'Jane', 'Smith', 'jane.smith@example.com'),
  (10, 'Richard', 'Johnson', 'richard.johnson@example.com')""")

## Kafka Source Data

In [ ]:
message_payload = '{"Mortgage_id": "123", "Mortgage_fac": "M1-str", "Mortgage_score": 30}'
spark.sql(f"INSERT INTO {feature_schema}.src_kafka_sink_sample_source (Message_payload) VALUES ('{message_payload}')")